In [3]:
from dotenv import load_dotenv
load_dotenv()  

True

In [4]:
import torch

In [5]:
from datasets import load_dataset

data = load_dataset("open-r1/Mixture-of-Thoughts", "science", split="train")

In [6]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-Embedding-4B")
encoder = AutoModel.from_pretrained("Qwen/Qwen3-Embedding-4B")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [7]:
encoder.to("cuda")

Qwen3Model(
  (embed_tokens): Embedding(151665, 2560)
  (layers): ModuleList(
    (0-35): 36 x Qwen3DecoderLayer(
      (self_attn): Qwen3Attention(
        (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
        (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
        (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
        (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
        (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
        (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
      )
      (mlp): Qwen3MLP(
        (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
        (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
        (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
      (post_attention_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
    )
  )
  (norm): Qwen3RM

In [8]:
instructions = [d['messages'][0]['content'] for d in data]
instructions

['What are examples of food-safe polyelectrolytes that act as bases and develop positive charges along the polymer?A: glucosamine, chitin, chitosan\nB: xylan, mannan, galactan\nC: pectin, alginate, carrageenan\nD: cellulose, lignin, hemi cellulose',
 'What are examples of food-safe polyelectrolytes that act as bases and develop positive charges along the polymer?A: glucosamine, chitin, chitosan\nB: xylan, mannan, galactan\nC: pectin, alginate, carrageenan\nD: cellulose, lignin, hemi cellulose',
 'What are examples of food-safe polyelectrolytes that act as bases and develop positive charges along the polymer?A: glucosamine, chitin, chitosan\nB: xylan, mannan, galactan\nC: pectin, alginate, carrageenan\nD: cellulose, lignin, hemi cellulose',
 'What is the hybridization of the carbon atom in the carbanion $\\ce{R3C-}$?A: sp^3\nB: sp^3d\nC: sp^2\nD: sp',
 'What is the hybridization of the carbon atom in the carbanion $\\ce{R3C-}$?A: sp^3\nB: sp^3d\nC: sp^2\nD: sp',
 'What is the hybridizat

In [9]:
tokenized = tokenizer(instructions[:20], padding=True, truncation=True, return_tensors="pt")
tokenized.to(encoder.device)
tokenized

{'input_ids': tensor([[  3838,    525,  10295,  ...,  68902,    960, 151643],
        [  3838,    525,  10295,  ...,  68902,    960, 151643],
        [  3838,    525,  10295,  ...,  68902,    960, 151643],
        ...,
        [  6713,  54893,  83343,  ..., 151643, 151643, 151643],
        [  3838,    374,    279,  ..., 151643, 151643, 151643],
        [  3838,    374,    279,  ..., 151643, 151643, 151643]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]], device='cuda:0')}

In [10]:
result = encoder(**tokenized)
result

BaseModelOutputWithPast(last_hidden_state=tensor([[[ 0.1514,  5.3125, -1.0938,  ...,  3.2500, -4.4375, -0.3711],
         [-0.0679,  7.0000,  9.8750,  ...,  0.6953,  1.0547,  1.2109],
         [-0.0742,  1.6406,  4.6875,  ...,  1.7109, -3.4531,  5.7188],
         ...,
         [-0.0879, -3.5312,  0.3438,  ...,  4.5625,  0.3223, -0.3496],
         [-0.0625, -1.7109,  2.4375,  ...,  1.0156, -1.5000,  0.1504],
         [-0.0415, -0.4199, -7.5938,  ...,  0.1992,  0.8945, -2.5469]],

        [[ 0.1514,  5.3125, -1.0938,  ...,  3.2500, -4.4375, -0.3711],
         [-0.0679,  7.0000,  9.8750,  ...,  0.6953,  1.0547,  1.2109],
         [-0.0742,  1.6406,  4.6875,  ...,  1.7109, -3.4531,  5.7188],
         ...,
         [-0.0879, -3.5312,  0.3438,  ...,  4.5625,  0.3223, -0.3496],
         [-0.0625, -1.7109,  2.4375,  ...,  1.0156, -1.5000,  0.1504],
         [-0.0415, -0.4199, -7.5938,  ...,  0.1992,  0.8945, -2.5469]],

        [[ 0.1514,  5.3125, -1.0938,  ...,  3.2500, -4.4375, -0.3711],
   

In [31]:
result.last_hidden_state.shape

torch.Size([20, 75, 2560])

In [11]:
pooled = result.last_hidden_state[:, -1, :] #eos pooling
pooled

tensor([[-0.0415, -0.4199, -7.5938,  ...,  0.1992,  0.8945, -2.5469],
        [-0.0415, -0.4199, -7.5938,  ...,  0.1992,  0.8945, -2.5469],
        [-0.0415, -0.4199, -7.5938,  ...,  0.1992,  0.8945, -2.5469],
        ...,
        [ 0.0330, -4.3125,  0.2910,  ...,  3.2344,  1.1172, -3.8438],
        [ 0.0537, -1.4219,  7.5312,  ..., -0.6133,  1.0781,  3.7969],
        [ 0.0537, -1.4219,  7.5312,  ..., -0.6133,  1.0781,  3.7969]],
       device='cuda:0', dtype=torch.bfloat16, grad_fn=<SelectBackward0>)

In [32]:
pooled.shape

torch.Size([20, 2560])

In [12]:
from transformers import AutoModelForCausalLM

In [13]:
llm = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-4B")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [14]:
llm.to("cuda")

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (post_attention_layer

In [15]:
response = llm(tokenized.input_ids[0].unsqueeze(0), attention_mask=tokenized.attention_mask[0].unsqueeze(0))
response

CausalLMOutputWithPast(loss=None, logits=tensor([[[ 7.0625, 15.5000,  7.8438,  ..., -1.2188, -1.2188, -1.2188],
         [-2.7812,  0.8438, -1.2188,  ...,  1.8906,  1.8906,  1.8906],
         [ 8.5000,  8.2500,  5.2500,  ...,  2.3594,  2.3594,  2.3594],
         ...,
         [ 5.8750,  6.2812,  2.7031,  ..., -5.0625, -5.0625, -5.0625],
         [11.1250, 13.0000,  8.9375,  ...,  3.3750,  3.3750,  3.3750],
         [10.9375, 14.6250, 16.0000,  ...,  3.3750,  3.3750,  3.3750]]],
       device='cuda:0', dtype=torch.bfloat16, grad_fn=<UnsafeViewBackward0>), past_key_values=DynamicCache(layers=[DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLayer, DynamicLay

In [33]:
response.logits.shape

torch.Size([1, 75, 151936])

In [17]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebook":   # kernel was started from notebook/
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

In [18]:
from torch.utils.data.dataloader import DataLoader
from src.data.collate import pad_collate
from src.data.openr1_reasoning_dataset import ReasoningDataset

dataset = ReasoningDataset(data, tokenizer, step_separator="\n\n")
dataloader = DataLoader(dataset, batch_size=2, collate_fn=pad_collate)

In [19]:
dataloader_iter = iter(dataloader)
batch = next(dataloader_iter)

In [20]:
batch["ctx_input_ids"].shape, batch["step_input_ids"].shape

(torch.Size([2, 75]), torch.Size([2, 11, 256]))

In [27]:
B, S, L = batch["step_input_ids"].shape          # (2, 11, 256)

step_ids = batch["step_input_ids"].to(encoder.device).reshape(B * S, L)
step_mask = batch["step_attention_mask"].to(encoder.device).reshape(B * S, L)

with torch.no_grad():
    h = encoder(input_ids=step_ids, attention_mask=step_mask).last_hidden_state
    # h: [B*S, L, D] = [22, 256, 2560]

h = h.reshape(B, S, L, -1)                        # back to per-step

In [29]:
h.shape

torch.Size([2, 11, 256, 2560])